In [1]:
import pandas as pd
import zipfile
import os

# --- 1. List of your FAERS ZIP files ---
zip_files = [
    "faers_ascii_2024Q1.zip",
    "faers_ascii_2024Q2.zip"]

# --- 2. Columns to drop ---
unwanted_cols = [
    'val_vbm', 'dose_vbm', 'cum_dose_chr', 'cum_dose_unit',
    'dechal', 'rechal', 'lot_num', 'exp_dt', 'nda_num'
]

# --- 3. Drug names to keep ---
drug_list = [
    "Lamotrigine", "Levetiracetam", "Topiramate", "Gabapentin", "Pregabalin",
    "Oxcarbazepine", "Zonisamide", "Lacosamide", "Clobazam", "Phenytoin",
    "Carbamazepine", "Phenobarbital", "Valproic acid", "Sodium valproate",
    "Ethosuximide", "Levodopa + Carbidopa", "Bromocriptine", "Pramipexole",
    "Ropinirole", "Rotigotine", "Apomorphine", "Selegiline", "Rasagiline",
    "Safinamide", "Entacapone", "Tolcapone", "Trihexyphenidyl", "Benzhexol",
    "Benztropine", "Amantadine", "Donepezil", "Rivastigmine", "Galantamine",
    "Memantine", "Interferon beta-1a", "Interferon beta-1b", "Glatiramer acetate",
    "Fingolimod", "Teriflunomide", "Dimethyl fumarate", "Natalizumab",
    "Ocrelizumab", "Alemtuzumab", "Baclofen", "Tizanidine", "Modafinil",
    "Sumatriptan", "Rizatriptan", "Zolmitriptan", "Ibuprofen", "Naproxen",
    "Ergotamine", "Dihydroergotamine", "Metoclopramide", "Domperidone",
    "Propranolol", "Amitriptyline", "Candesartan", "Botulinum toxin A",
    "Nortriptyline", "Duloxetine", "Tetrabenazine", "Deutetrabenazine",
    "Haloperidol", "Risperidone", "Diazepam", "Pyridostigmine", "Neostigmine",
    "Prednisolone", "Azathioprine", "Mycophenolate mofetil", "Cyclosporine",
    "Eculizumab", "Rituximab", "Plasmapheresis", "IV immunoglobulin",
    "Zolpidem", "Zopiclone", "Melatonin", "Sodium oxybate", "Methylphenidate",
    "Ceftriaxone", "Vancomycin", "Acyclovir", "Amphotericin B", "Citicoline",
    "Piracetam", "Cerebrolysin", "Edaravone"
]



In [2]:
# Lowercase version for case-insensitive comparison
drug_list_lower = [d.strip().lower() for d in drug_list]

# --- Master DataFrame ---
master_drug_df = pd.DataFrame()   # <- initialize here

# --- Loop through ZIP files ---
for zip_path in zip_files:
    if not os.path.exists(zip_path):
        print(f"File not found: {zip_path}")
        continue
    
    with zipfile.ZipFile(zip_path, "r") as z:
        # Find DRUG tables
        drug_files = [f for f in z.namelist() if "DRUG" in f.upper() and f.endswith(".txt")]
        
        for file_name in drug_files:
            with z.open(file_name) as f:
                df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)
                
                # Drop unwanted columns if they exist
                df = df.drop(columns=[c for c in unwanted_cols if c in df.columns])
                
                # Filter only if required columns exist
                if {'drugname', 'role_cod'}.issubset(df.columns):
                    df = df[
                        (df['drugname'].str.strip().str.lower().isin(drug_list_lower)) &
                        (df['role_cod'].isin(['PS', 'SS']))
                    ]
                    
                    # Append filtered rows
                    master_drug_df = pd.concat([master_drug_df, df], ignore_index=True)


In [3]:
#  REMOVE DUPLICATES
master_drug_df = master_drug_df.drop_duplicates(
    subset=['primaryid', 'caseid', 'drugname']
)


In [4]:
# Convert 'Unknown' values to NaN
master_drug_df = master_drug_df.replace('Unknown', np.nan)

# Check percentage of missing values per column
missing_percent = master_drug_df.isna().mean() * 100
print("Missing value percentages per column:\n", missing_percent)

# Drop columns where more than 60% of values are missing
cols_to_drop = missing_percent[missing_percent > 60].index
print(f"Columns to drop (more than 60% missing): {list(cols_to_drop)}")

master_drug_df_clean = master_drug_df.drop(columns=cols_to_drop)

# Optional: Check the updated DataFrame
print("Remaining columns:", master_drug_df_clean.columns)


Missing value percentages per column:
 primaryid     0.000000
caseid        0.000000
drug_seq      0.000000
role_cod      0.000000
drugname      0.000000
prod_ai       0.001435
route        69.344563
dose_amt     66.566706
dose_unit    66.566706
dose_form    75.523001
dose_freq    82.971274
dtype: float64
Columns to drop (more than 60% missing): ['route', 'dose_amt', 'dose_unit', 'dose_form', 'dose_freq']
Remaining columns: Index(['primaryid', 'caseid', 'drug_seq', 'role_cod', 'drugname', 'prod_ai'], dtype='object')


In [5]:
print(f"Total rows after filtering: {len(master_drug_df)}")
master_drug_df_clean.head(100)

Total rows after filtering: 69694


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE
2,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE
4,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL
6,101823182,10182318,2,SS,PRAMIPEXOLE,PRAMIPEXOLE\PRAMIPEXOLE DIHYDROCHLORIDE
7,103543884,10354388,2,SS,DIAZEPAM,DIAZEPAM
...,...,...,...,...,...,...
184,121237232,12123723,5,SS,AMITRIPTYLINE,AMITRIPTYLINE
185,121237402,12123740,3,SS,DIAZEPAM,DIAZEPAM
186,121237432,12123743,2,SS,DIAZEPAM,DIAZEPAM
187,121237512,12123751,5,SS,NORTRIPTYLINE,NORTRIPTYLINE


In [6]:
# --- Master REAC DataFrame ---
master_reac_df = pd.DataFrame()  # <- fix this line

# --- Read and clean REAC tables ---
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        reac_files = [f for f in z.namelist() if "REAC" in f.upper() and f.endswith(".txt")]

        for file_name in reac_files:
            with z.open(file_name) as f:
                reac_df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)

                # Remove drug_rec_act column if exists
                reac_df = reac_df.drop(columns=['drug_rec_act'], errors='ignore')

                # Keep only join-relevant columns
                reac_df = reac_df[['primaryid', 'pt']]

                master_reac_df = pd.concat([master_reac_df, reac_df], ignore_index=True)

# --- Join DRUG + REAC ---
final_df = master_drug_df_clean.merge(
    master_reac_df,
    on='primaryid',
    how='inner'
)

print("Final dataset shape:", final_df.shape)
final_df.head(10)


Final dataset shape: (665723, 7)


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,pt
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Toxicity to various agents
1,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Mycobacterium haemophilum infection
2,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Wound infection staphylococcal
3,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Staphylococcal infection
4,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Toxicity to various agents
5,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Mycobacterium haemophilum infection
6,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Wound infection staphylococcal
7,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Staphylococcal infection
8,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Toxicity to various agents
9,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Mycobacterium haemophilum infection


In [7]:
import pandas as pd
import zipfile

# Loop through ZIP files and read DEMO table
for zip_path in zip_files:
    with zipfile.ZipFile(zip_path, "r") as z:
        # Find DEMO file
        demo_files = [f for f in z.namelist() if "DEMO" in f.upper() and f.endswith(".txt")]
        
        if demo_files:
            with z.open(demo_files[0]) as f:
                demo_df = pd.read_csv(f, sep="$", encoding="latin-1", low_memory=False)
                
                print(f"\nFirst 10 records from DEMO table in {zip_path}:")  
                break  # stop after first DEMO table



First 10 records from DEMO table in faers_ascii_2024Q1.zip:


In [8]:
# Columns to drop manually
demo_cols_to_drop = [
    'rept_cod', 'to_mfr', 'caseversion', 'i_f_code', 'event_dt',
    'mfr_dt', 'init_fda_dt', 'fda_dt', 'rept_dt', 'occp_cod','reporter_country','e_sub'
]

# Drop manually specified columns (if they exist)
demo_df_clean = demo_df.drop(
    columns=[col for col in demo_cols_to_drop if col in demo_df.columns]
)

# Drop columns where >60% values are null
threshold = 0.6  # 60% null
demo_df_clean = demo_df_clean.dropna(axis=1, thresh=int((1 - threshold) * len(demo_df_clean)))

print("Remaining DEMO columns after cleaning:")
print(demo_df_clean.columns)

# --- Join FINAL with DEMO ---
# Use suffixes to avoid automatic _x/_y column names
final_demo_df = final_df.merge(
    demo_df_clean,
    on='primaryid',
    how='inner',
    suffixes=('_drug', '_demo')  # rename overlapping columns clearly
)

# Optional: drop duplicate columns if not needed (example: caseid_demo)
if 'caseid_demo' in final_demo_df.columns:
    final_demo_df = final_demo_df.drop(columns=['caseid_demo'])

# Optional: rename columns for clarity
if 'caseid_drug' in final_demo_df.columns:
    final_demo_df = final_demo_df.rename(columns={'caseid_drug': 'caseid'})

print("Final DRUG + REAC + DEMO dataset shape:", final_demo_df.shape)
final_demo_df.head(10)

Remaining DEMO columns after cleaning:
Index(['primaryid', 'caseid', 'mfr_num', 'mfr_sndr', 'age', 'age_cod', 'sex',
       'occr_country'],
      dtype='object')
Final DRUG + REAC + DEMO dataset shape: (302569, 13)


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,pt,mfr_num,mfr_sndr,age,age_cod,sex,occr_country
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
1,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
2,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Wound infection staphylococcal,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
3,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Staphylococcal infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
4,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
5,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
6,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Wound infection staphylococcal,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
7,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Staphylococcal infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
8,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Toxicity to various agents,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU
9,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Mycobacterium haemophilum infection,PHHY2014AU032943,NOVARTIS,32.0,YR,M,AU


In [9]:
# --- List of your FAERS ZIP files ---
zip_files = [
    "faers_ascii_2024Q1.zip",
    "faers_ascii_2024Q2.zip",
]

# --- Function to read a FAERS table from all ZIPs ---
def read_faers_table(table_keyword):
    all_tables = []
    for zip_path in zip_files:
        with zipfile.ZipFile(zip_path, "r") as z:
            table_files = [f for f in z.namelist() if table_keyword.upper() in f.upper() and f.endswith(".txt")]
            for file_name in table_files:
                df = pd.read_csv(z.open(file_name), sep='$', dtype=str, encoding='latin1', low_memory=False)
                all_tables.append(df)
    if all_tables:
        return pd.concat(all_tables, ignore_index=True)
    else:
        print(f"No tables found for keyword: {table_keyword}")
        return pd.DataFrame()

# --- Read OUTC table ---
outc_df = read_faers_table("OUTC")


In [10]:
# --- Ensure consistent types ---
final_demo_df['primaryid'] = final_demo_df['primaryid'].astype(str)
outc_df['primaryid'] = outc_df['primaryid'].astype(str)

# --- Drop manufacturer-related columns (not analytically useful) ---
final_demo_df = final_demo_df.drop(
    columns=[c for c in ['mfr_num', 'mfr_sndr'] if c in final_demo_df.columns],
    errors='ignore'
)

# --- Remove existing outcome columns (important for reruns) ---
final_demo_df = final_demo_df.drop(
    columns=[c for c in final_demo_df.columns if c.startswith('outc_cod')],
    errors='ignore'
)

# --- Aggregate OUTC (one row per primaryid) ---
outc_agg = (
    outc_df
    .groupby('primaryid')['outc_cod']
    .apply(lambda x: ','.join(sorted(x.unique())))
    .reset_index()
)

# --- Merge aggregated OUTC ---
final_demo_df = final_demo_df.merge(
    outc_agg,
    on='primaryid',
    how='left'
)

# --- Check result ---
print("Final dataset shape:", final_demo_df.shape)
final_demo_df.head(10)


Final dataset shape: (302569, 12)


,primaryid,caseid,drug_seq,role_cod,drugname,prod_ai,pt,age,age_cod,sex,occr_country,outc_cod
0,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Toxicity to various agents,32.0,YR,M,AU,OT
1,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Mycobacterium haemophilum infection,32.0,YR,M,AU,OT
2,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Wound infection staphylococcal,32.0,YR,M,AU,OT
3,100293663,10029366,1,PS,CYCLOSPORINE,CYCLOSPORINE,Staphylococcal infection,32.0,YR,M,AU,OT
4,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Toxicity to various agents,32.0,YR,M,AU,OT
5,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Mycobacterium haemophilum infection,32.0,YR,M,AU,OT
6,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Wound infection staphylococcal,32.0,YR,M,AU,OT
7,100293663,10029366,5,SS,PREDNISOLONE,PREDNISOLONE,Staphylococcal infection,32.0,YR,M,AU,OT
8,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Toxicity to various agents,32.0,YR,M,AU,OT
9,100293663,10029366,7,SS,MYCOPHENOLATE MOFETIL,MYCOPHENOLATE MOFETIL,Mycobacterium haemophilum infection,32.0,YR,M,AU,OT


In [11]:

import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

#  Prepare dataframe 
df = final_demo_df[['caseid', 'drugname', 'pt', 'outc_cod']].copy()

# Standardize names
df['drugname'] = df['drugname'].str.lower().str.strip()
df['pt'] = df['pt'].str.lower().str.strip()

# Ensure outc_cod is string and fill missing values
df['outc_cod'] = df['outc_cod'].astype(str).fillna('')

# Create transaction items
drug_items = df[['caseid', 'drugname']].rename(columns={'drugname': 'item'})
pt_items = df[['caseid', 'pt']].rename(columns={'pt': 'item'})

txn_items = pd.concat([drug_items, pt_items], ignore_index=True)
txn_items.drop_duplicates(inplace=True)

# Group by caseid to create baskets
transactions = (
    txn_items.groupby('caseid')['item']
    .apply(list)
    .reset_index(name='basket')
)

# Encode for FP-Growth
te = TransactionEncoder()
te_array = te.fit(transactions['basket']).transform(transactions['basket'])
basket_df = pd.DataFrame(te_array, columns=te.columns_).astype(bool)

#  Run FP-Growth 
frequent_itemsets = fpgrowth(
    basket_df,
    min_support=0.001,  # adjust if needed
    use_colnames=True,
    max_len=2           # Only pairs (Drug + PT)
)

#  Generate association rules
rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.2
)

#  Filter only Drug → PT rules 
drug_set = set(df['drugname'].unique())
pt_set = set(df['pt'].unique())

drug_pt_rules = rules[
    rules['antecedents'].apply(lambda x: set(x).issubset(drug_set)) &
    rules['consequents'].apply(lambda x: set(x).issubset(pt_set))
].reset_index(drop=True)

#  Prepare rules for merging outcome counts
# Convert frozensets to string
drug_pt_rules['drugname'] = drug_pt_rules['antecedents'].apply(lambda x: list(x)[0])
drug_pt_rules['pt'] = drug_pt_rules['consequents'].apply(lambda x: list(x)[0])

# Keep only needed columns
drug_pt_rules = drug_pt_rules[['drugname', 'pt', 'support', 'confidence', 'lift']]

#  Aggregate outcomes per drug–PT pair
outcome_agg = (
    df.groupby(['drugname', 'pt'])
      .agg(
          reports=('caseid', 'nunique'),   # total supporting cases
          outc_cod=('outc_cod', lambda x: ','.join(sorted(set([str(i) for i in x if i]))))  # safe aggregation
      )
      .reset_index()
)

#  Merge aggregated outcomes with rules
final_rules_with_outcomes = drug_pt_rules.merge(
    outcome_agg,
    on=['drugname', 'pt'],
    how='left'
)

#  Sort 
final_rules_with_outcomes = final_rules_with_outcomes.sort_values(by='lift', ascending=False).reset_index(drop=True)



In [12]:
#  View final rules
final_rules_with_outcomes.head(20)


,drugname,pt,support,confidence,lift,reports,outc_cod
0,domperidone,amaurosis fugax,0.001358,0.261905,151.511905,33,"DE,OT,OT"
1,domperidone,urinary tract disorder,0.001358,0.261905,138.336957,33,"DE,OT,OT"
2,donepezil,amaurosis fugax,0.001441,0.212121,122.712121,35,"DE,OT,OT"
3,donepezil,haemorrhagic stroke,0.001893,0.278788,116.788088,46,"DE,DE,OT"
4,donepezil,urinary tract disorder,0.001441,0.212121,112.041502,35,"DE,OT,OT"
5,domperidone,presyncope,0.001317,0.253968,106.390805,32,"DE,OT,OT"
6,domperidone,ocular discomfort,0.001317,0.253968,104.587571,32,"DE,OT,OT"
7,domperidone,haemorrhagic stroke,0.001276,0.246032,103.066092,31,"DE,OT"
8,donepezil,ocular discomfort,0.001605,0.236364,97.337750,39,"DE,OT,OT"
9,sodium oxybate,product administration interrupted,0.001687,0.336066,97.206967,41,"DE,OT,HO,HO,OT,OT,nan"


In [13]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth


#  Prepare the dataset
df = final_demo_df[['caseid', 'drugname', 'pt']].copy()

# Clean strings
df['drugname'] = df['drugname'].str.lower().str.strip()
df['pt'] = df['pt'].str.lower().str.strip()


# Create transaction baskets (all drugs + ADRs)
drug_items = df[['caseid', 'drugname']].rename(columns={'drugname': 'item'})
pt_items = df[['caseid', 'pt']].rename(columns={'pt': 'item'})

txn_items = pd.concat([drug_items, pt_items], ignore_index=True)
txn_items.drop_duplicates(inplace=True)

# Group by caseid
transactions = (
    txn_items.groupby('caseid')['item']
    .apply(list)
    .reset_index(name='basket')
)


# Encode transactions
te = TransactionEncoder()
te_array = te.fit(transactions['basket']).transform(transactions['basket'])
basket_df = pd.DataFrame(te_array, columns=te.columns_)

# Run FP-Growth with 3 items
frequent_itemsets = fpgrowth(
    basket_df,
    min_support=0.001,  # adjust as needed
    use_colnames=True,
    max_len=3           # find triplets (Drug + Drug + ADR)
)


#  Filter to exactly 2 Drugs + 1 ADR
drug_set = set(df['drugname'].unique())
pt_set = set(df['pt'].unique())

def is_drug_drug_adr(itemset):
    drugs = [i for i in itemset if i in drug_set]
    adrs = [i for i in itemset if i in pt_set]
    return len(drugs) == 2 and len(adrs) == 1

frequent_itemsets['itemset_list'] = frequent_itemsets['itemsets'].apply(lambda x: list(x))
drug_drug_adr_sets = frequent_itemsets[
    frequent_itemsets['itemset_list'].apply(is_drug_drug_adr)
].reset_index(drop=True)

# Display results
print(drug_drug_adr_sets[['itemset_list', 'support']])


                                     itemset_list   support
0          [gabapentin, off label use, rituximab]  0.002799
1       [gabapentin, drug ineffective, rituximab]  0.001770
2          [gabapentin, off label use, ibuprofen]  0.002346
3       [gabapentin, drug ineffective, ibuprofen]  0.002140
4          [rituximab, pregabalin, off label use]  0.001687
...                                           ...       ...
2520     [pregabalin, donepezil, amaurosis fugax]  0.001441
2521    [donepezil, amaurosis fugax, propranolol]  0.001358
2522   [amaurosis fugax, pregabalin, domperidone]  0.001235
2523    [donepezil, amaurosis fugax, domperidone]  0.001111
2524  [amaurosis fugax, domperidone, propranolol]  0.001276

[2525 rows x 2 columns]


In [14]:
from mlxtend.frequent_patterns import association_rules

# Generate rules
rules = association_rules(
    frequent_itemsets,       # all frequent itemsets from FP-Growth
    metric='confidence',     # metric to evaluate rules
    min_threshold=0.2        # minimum confidence
)

# Filter to only keep Drug + Drug -> ADR rules
drug_set = set(df['drugname'].unique())
pt_set = set(df['pt'].unique())

def is_drug_drug_to_adr_rule(row):
    antecedent = row['antecedents']
    consequent = row['consequents']
    return (
        len(antecedent & drug_set) == 2 and   # 2 drugs in antecedent
        len(consequent & pt_set) == 1         # 1 ADR in consequent
    )

drug_drug_adr_rules = rules[rules.apply(is_drug_drug_to_adr_rule, axis=1)].reset_index(drop=True)

# Optional: convert frozensets to strings for readability
drug_drug_adr_rules['antecedents_str'] = drug_drug_adr_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
drug_drug_adr_rules['consequents_str'] = drug_drug_adr_rules['consequents'].apply(lambda x: ', '.join(list(x)))

# Keep relevant columns
drug_drug_adr_rules = drug_drug_adr_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']]


In [15]:
drug_drug_adr_rules

,antecedents_str,consequents_str,support,confidence,lift
0,"gabapentin, rituximab",off label use,0.002799,0.581197,3.775758
1,"gabapentin, rituximab",drug ineffective,0.001770,0.367521,2.637232
2,"gabapentin, ibuprofen",off label use,0.002346,0.612903,3.981741
3,"gabapentin, ibuprofen",drug ineffective,0.002140,0.559140,4.012233
4,"rituximab, pregabalin",off label use,0.001687,0.422680,2.745953
...,...,...,...,...,...
2288,"pregabalin, donepezil",amaurosis fugax,0.001441,0.625000,361.562500
2289,"donepezil, propranolol",amaurosis fugax,0.001358,0.970588,561.485294
2290,"pregabalin, domperidone",amaurosis fugax,0.001235,0.769231,445.000000
2291,"donepezil, domperidone",amaurosis fugax,0.001111,0.964286,557.839286


In [16]:

# Prepare single-drug → ADR lifts
# Filter rules for 1 Drug -> ADR
single_drug_to_adr = rules[rules.apply(lambda row: len(row['antecedents'] & drug_set)==1 
                                       and len(row['consequents'] & pt_set)==1, axis=1)]

# Create a lookup dictionary for fast access
single_lift_lookup = {}
for idx, row in single_drug_to_adr.iterrows():
    drug = list(row['antecedents'])[0]
    adr = list(row['consequents'])[0]
    single_lift_lookup[(drug, adr)] = row['lift']


# Check 3-itemsets for interaction
interaction_signals = []

for idx, row in drug_drug_adr_sets.iterrows():
    drugs = [i for i in row['itemset_list'] if i in drug_set]
    adr = [i for i in row['itemset_list'] if i in pt_set][0]
    
    # Lift of each single-drug → ADR
    lift_A = single_lift_lookup.get((drugs[0], adr), 0)
    lift_B = single_lift_lookup.get((drugs[1], adr), 0)
    
    # Lift of 2-drugs → ADR (from association rules)
    # Find the rule {DrugA, DrugB} -> ADR
    mask = (drug_drug_adr_rules['antecedents_str'].apply(lambda x: set(x.split(', ')) == set(drugs))) & \
           (drug_drug_adr_rules['consequents_str'] == adr)
    
    if mask.any():
        lift_AB = drug_drug_adr_rules.loc[mask, 'lift'].values[0]
    else:
        lift_AB = row['support']  # fallback if rule not generated
    
    # If 2-drug lift >> max(single-drug lift), mark as interaction
    if lift_AB > max(lift_A, lift_B):
        interaction_signals.append({
            'DrugA': drugs[0],
            'DrugB': drugs[1],
            'ADR': adr,
            'Lift_2Drugs': lift_AB,
            'Lift_DrugA': lift_A,
            'Lift_DrugB': lift_B,
            'Support_3Itemset': row['support']
        })

# Convert to DataFrame
interaction_signals_df = pd.DataFrame(interaction_signals)



In [17]:
interaction_signals_df

,DrugA,DrugB,ADR,Lift_2Drugs,Lift_DrugA,Lift_DrugB,Support_3Itemset
0,levetiracetam,valproic acid,drug ineffective,1.861809,0.000000,0.000000,0.001976
1,lamotrigine,valproic acid,treatment failure,12.049095,0.000000,0.000000,0.001729
2,levetiracetam,valproic acid,treatment failure,9.467146,0.000000,0.000000,0.002264
3,amitriptyline,diazepam,toxicity to various agents,16.431695,0.000000,5.687273,0.001358
4,naproxen,amitriptyline,drug ineffective,5.075512,3.454978,4.783816,0.001194
...,...,...,...,...,...,...,...
136,topiramate,clobazam,multiple-drug resistance,62.992222,0.000000,0.000000,0.001152
137,levetiracetam,clobazam,multiple-drug resistance,0.001070,0.000000,0.000000,0.001070
138,levetiracetam,valproic acid,multiple-drug resistance,0.001194,0.000000,0.000000,0.001194
139,pregabalin,propranolol,presyncope,314.185345,185.594718,172.156353,0.001482


In [18]:
def assign_severity(row):
    if row['Support_3Itemset'] >= 0.002 and row['Lift_2Drugs'] >= 10:
        return 'Severe'
    elif row['Lift_2Drugs'] >= 10:   # strong signal, even if support is low
        return 'Severe (rare)'
    elif row['Support_3Itemset'] >= 0.001 or row['Lift_2Drugs'] >= 5:
        return 'Moderate'
    else:
        return 'Mild'

interaction_signals_df['Severity'] = interaction_signals_df.apply(assign_severity, axis=1)


In [19]:
interaction_signals_df.head(50)

,DrugA,DrugB,ADR,Lift_2Drugs,Lift_DrugA,Lift_DrugB,Support_3Itemset,Severity
0,levetiracetam,valproic acid,drug ineffective,1.861809,0.000000,0.000000,0.001976,Moderate
1,lamotrigine,valproic acid,treatment failure,12.049095,0.000000,0.000000,0.001729,Severe (rare)
2,levetiracetam,valproic acid,treatment failure,9.467146,0.000000,0.000000,0.002264,Moderate
3,amitriptyline,diazepam,toxicity to various agents,16.431695,0.000000,5.687273,0.001358,Severe (rare)
4,naproxen,amitriptyline,drug ineffective,5.075512,3.454978,4.783816,0.001194,Moderate
5,topiramate,amitriptyline,drug ineffective,5.406367,4.415830,4.783816,0.002264,Moderate
6,amitriptyline,ibuprofen,drug ineffective,4.976712,4.783816,3.348671,0.001770,Moderate
7,mycophenolate mofetil,ibuprofen,pyrexia,12.971248,8.153356,10.191695,0.001441,Severe (rare)
8,lamotrigine,azathioprine,drug ineffective,6.712774,4.783816,3.792882,0.001194,Moderate
9,mycophenolate mofetil,cyclosporine,cytomegalovirus infection reactivation,0.001029,0.000000,0.000000,0.001029,Moderate


In [20]:
interaction_signals_df.to_csv("interaction_signals.csv", index=False)
